[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q4/01_within_host_classifier.ipynb)

# R2-Q4 Week 2 — Train and evaluate the within-host classifier

### This notebook produces `classifier.pkl`, the trained model the cross-host evaluation runs on next week.

This notebook trains a disease classifier on PlantVillage following the decisions you precommitted in Week 1. The training runs within-host — the classifier sees disease examples from the hosts your precommit retained for training and gets evaluated on held-out samples from those same hosts. The within-host accuracy you measure here is the reference number the cross-host evaluation gets compared against next week.

By the end of this notebook you will have:

- A trained classifier saved as `classifier.pkl`, ready for the Week 3 cross-host evaluation.
- The within-host held-out test split saved as `within_host_test.parquet`, so the evaluation is reproducible.
- A within-host accuracy report saved as `within_host_accuracy.parquet`, with per-disease accuracy and the overall figure.
- Submitted EQ#2.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R2-Q4 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q4")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of `precommit.json` (so the disease list and hold-out scheme drive everything downstream)

Last week you locked three decisions in NB00 and wrote them to `precommit.json`: the **disease list** (which diseases your classifier will learn to tell apart), the **hold-out scheme** (which host plant is withheld for each disease, so it can be the cross-host test next week), and the **aggregation level** (how the cross-host accuracy gets reported). This week you train the classifier and measure how well it does *within* host — that is, on held-out images of the same host plants it trained on.

The first thing this notebook does is read those three decisions back. Two of them drive everything below:

- `disease_list` — the diseases your classifier learns. This is the label space: the model is trained to choose among these diseases, not among PlantVillage's full set of host-and-disease classes. (Section 2 explains why the label space has to be the disease, not the host-plus-disease pair.)
- `hold_out_scheme` — for each disease, which host is held out. Section 2 trains the classifier on every *other* host for that disease, and measures within-host accuracy on a held-out slice of those same training hosts. The withheld host is set aside untouched for next week's cross-host test, so the within-host test you run here and the cross-host test you run in NB02 are carved from the same scheme and stay comparable.

The reason to *load* these rather than retype them is the whole point of having precommitted: the classifier you train this week should be driven by the choices you reasoned about and locked in Week 1, not by fresh choices made after you start seeing numbers. If you find yourself wanting to change a precommit value now, that's a change to your methodology — make it in NB00 and re-run, so the record stays honest.

In [ ]:
# Read back the three decisions you locked in NB00. The load is defensive: a
# missing or half-written file should fail here, with a clear message, rather
# than partway through training. NB01 drives off two of the three fields
# (disease_list, hold_out_scheme); aggregation is read to keep the within-host
# report's granularity matched to next week's cross-host report.
import json

precommit_path = OUTPUT_DIR / "precommit.json"
if not precommit_path.exists():
    raise FileNotFoundError(
        f"No precommit.json at {precommit_path}. "
        "Run NB00 (00_orient_and_precommit.ipynb) to completion first — "
        "it writes the Week-1 decisions this notebook reads."
    )

with open(precommit_path) as f:
    precommit = json.load(f)

# Every field should be populated before Week 2 starts.
for key in ("disease_list", "hold_out_scheme", "aggregation"):
    if precommit.get(key) is None:
        raise KeyError(
            f"precommit.json is missing '{key}'. Re-run NB00 — every field "
            "should be filled in before you start Week 2."
        )

diseases    = precommit["disease_list"]["diseases"]
scheme      = precommit["hold_out_scheme"]["scheme"]
held_out    = precommit["hold_out_scheme"]["held_out_host"]
agg_level   = precommit["aggregation"]["level"]

# NB01 trains ONE classifier under a single fixed host-assignment. A
# leave-one-host-out scheme trains one classifier per held-out host per disease,
# which this notebook isn't built to loop over — so flag it rather than quietly
# running the wrong shape. (This is the same logic NB00 used to default the
# scheme to fixed_host.)
if scheme != "fixed_host":
    print(
        f"WARNING: hold_out_scheme is {scheme!r}, not 'fixed_host'. This "
        "notebook trains a single classifier under one fixed host assignment; "
        "leave-one-host-out needs one classifier per held-out configuration "
        "and a different loop. Check with your mentor before continuing."
    )

# Every disease in the list needs a withheld host named for it, or there's no
# cross-host test to set up next week.
missing_hosts = [d for d in diseases if d not in held_out]
if missing_hosts:
    raise KeyError(
        f"hold_out_scheme.held_out_host has no entry for {missing_hosts}. "
        "Every disease in your list needs a withheld host. Re-run NB00."
    )

# Print back what drives the rest of the notebook.
print("Loaded precommit.json")
print(f"  Diseases ({len(diseases)}): {', '.join(diseases)}")
print(f"  Hold-out scheme: {scheme}")
for d in diseases:
    print(f"    {d:18s} -> withhold host: {held_out[d]}")
print(f"  Aggregation level: {agg_level}")

Before training, set how this run executes. There's one knob, `DEV_MODE`, and it does two things at once: which size of PlantVillage to load, and how many training epochs to run.

Leave `DEV_MODE = True` while you're getting the notebook to run end to end. It uses a small slice of PlantVillage and a short two-epoch cap, so the whole path — loading data, carving the split, training, and measuring within-host accuracy — finishes in a few minutes and you catch mistakes cheaply. Switch to `DEV_MODE = False` for the real run: the full dataset and the full ten-epoch recipe, which takes tens of minutes on a GPU but produces the numbers you'll report and carry into next week.

The numbers from a `DEV_MODE = True` run are *not* the ones to report — a two-epoch model on a tiny slice tells you the notebook works, not how well the disease classifies. Keep that straight when you read the within-host accuracies in Section 4.

Training also requires a GPU. The cell below checks for one and stops early, with instructions, if it doesn't find it — a clearer failure than a cryptic error several minutes into training.

In [ ]:
# Run configuration — set once here; every section below reads from it.
#
# DEV_MODE is the one knob you flip while getting the notebook running:
#   True  -> the "tiny" PV variant (~50 images/class) and a 2-epoch cap.
#            The whole notebook finishes in a few minutes.
#   False -> the "full" PV variant and the full 10-epoch recipe. The real
#            run: tens of minutes on a GPU, and the numbers you report.
DEV_MODE = True

PV_VARIANT = "tiny" if DEV_MODE else "full"
EPOCH_CAP  = 2 if DEV_MODE else None   # None -> train_baseline's full 10 epochs

# One seed for the whole notebook. train_baseline also seeds itself per call
# from its own `seed=` argument; seeding here keeps anything outside that call
# (data loading, the within-host split, evaluation ordering) reproducible too.
iri.seed_all(42)

# train_baseline calls .cuda() directly, so a GPU is not optional here. Fail
# now, with a clear message, rather than partway through training.
assert iri.has_gpu(), (
    "No GPU detected. This notebook trains a ResNet-18 classifier and needs a "
    "GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run from "
    "the top."
)

print(f"DEV_MODE   = {DEV_MODE}")
print(f"PV_VARIANT = {PV_VARIANT!r}")
print(f"EPOCH_CAP  = {EPOCH_CAP}")
print("Seed set to 42; GPU detected.")

### 2) Load the PV slice per the precommitted disease list and hold-out scheme

Now load PlantVillage and shape it for the experiment. There are two jobs here, and the second one is the structural heart of this whole question.

The first job is ordinary: load the dataset and look at what came back. `load_plantvillage` returns a metadata table (one row per image, with columns for the host plant, the disease, and which split the image belongs to) alongside the images themselves.

The second job is the one to slow down on: deciding what the classifier's labels are.

In [ ]:
# Load PlantVillage at the size DEV_MODE selected. The loader returns a metadata
# table (one row per image) and the images. Do NOT reset_index on the metadata
# anywhere below — the frame's index is how PlantVillageDataset lines each row up
# with its image.
meta, hf = iri.load_plantvillage(variant=PV_VARIANT)

print(f"PlantVillage ({PV_VARIANT}): {len(meta):,} images")
print(f"Columns: {list(meta.columns)}")
print()
print("A few rows (host / disease / split):")
print(meta[["host", "disease", "split"]].head())

**What should the classifier predict?**

PlantVillage's built-in labels pair a host with a disease — `Tomato___Early_blight`, `Potato___Early_blight`, and so on. The obvious move would be to train on those labels directly. For this experiment, that's the wrong move, and the reason is worth being precise about.

You're going to withhold one host per disease and test on it next week. If the labels were host-and-disease pairs, the withheld host's label — say `Potato___Early_blight` — would never appear in training at all. The classifier couldn't predict a label it was never trained on, so "cross-host accuracy" would be undefined by construction. There would be nothing to measure.

So the label has to be the **disease alone**. `Tomato___Early_blight` and `Potato___Early_blight` both become just `Early_blight`. Now the same label is available whether the image came from a training host or the withheld host, and asking "does the model recognize early blight on a potato when it only ever saw it on tomatoes?" becomes a question you can actually score. That question — disease recognition that survives a change of host — is the entire point of R2-Q4.

Concretely: you'll keep only the rows for your precommitted diseases, drop the withheld host for each disease (those images are set aside for next week), and relabel the rest by disease. The classifier then learns to choose among your handful of diseases, and the withheld host becomes a fair test of whether it learned the disease or the plant.

**Within-host vs. cross-host, made concrete.** *Within-host* accuracy — what you measure this week — is the classifier's accuracy on held-out images of the *same* hosts it trained on. You get that hold-out for free from PlantVillage's own `test` split: those images were never in training, but they come from the training hosts. *Cross-host* accuracy — next week's job — is accuracy on the withheld host, which the classifier never saw at all. Both are measured on the same diseases; the gap between them is your finding.

**Practice 2.1 — apply your hold-out scheme.**

Build the training pool and the within-host test split. Three steps, and the middle one is the one that takes thought:

1. Keep only the rows whose disease is in your precommitted list. *(Given below.)*
2. For each disease, drop the rows whose host is the one you withheld. This is **not** one host dropped everywhere — different diseases can withhold different hosts, so it's a per-row comparison: for each image, is its host the host withheld *for that image's disease*?
3. Relabel by disease, so `class_idx` becomes the disease index rather than the host-and-disease class index. *(Given below, once step 2 is done.)*

Your fill is step 2: define `is_training_host`, a boolean Series that is `True` for rows whose host should be kept for training.

*Hint:* `in_scope["disease"].map(held_out)` gives, for each row, the host withheld for that row's disease. Compare that against `in_scope["host"]`.

In [ ]:
### 2.1) Apply the hold-out scheme
import pandas as pd

# disease -> 0..N-1. This is the label space: the classifier predicts among
# these diseases.
disease_to_idx = {d: i for i, d in enumerate(diseases)}

# Step 1 (given): rows for your precommitted diseases.
in_scope = meta[meta["disease"].isin(diseases)]

# Step 2 (your fill): define `is_training_host` — True where this row's host is
# NOT the host withheld for that row's disease. See the hint in the markdown
# above. (If you leave this blank, the next line raises NameError.)
# is_training_host = ...

train_hosts_frame = in_scope[is_training_host].copy()

# Step 3 (given): relabel to the disease index. Do NOT reset_index.
train_hosts_frame["class_idx"] = train_hosts_frame["disease"].map(disease_to_idx)

# The within-host test split is PlantVillage's own test split on these same
# (training) hosts — held out from training, but the same hosts.
within_host_test = train_hosts_frame[train_hosts_frame["split"] == "test"]

num_classes = len(diseases)
print(f"Training pool: {len(train_hosts_frame):,} images across {num_classes} diseases")
print(f"Within-host test split: {len(within_host_test):,} images")

In [ ]:
# Inspect before trusting. Three things have to hold before training is worth
# starting, and a quick table tells you whether your hold-out actually did what
# you meant.

# 1. The withheld host really is gone from the training pool, for every disease.
for d in diseases:
    rows = train_hosts_frame[train_hosts_frame["disease"] == d]
    assert held_out[d] not in set(rows["host"]), (
        f"{held_out[d]} is still in the training pool for {d} — the hold-out "
        "mask didn't drop it. Check your Step-2 comparison."
    )

# 2. The label space is exactly 0..N-1, with every disease present.
assert set(train_hosts_frame["class_idx"]) == set(range(num_classes)), (
    "The relabelled class_idx isn't a contiguous 0..N-1. Check disease_to_idx "
    "and the Step-3 map."
)

# 3. Every disease has within-host test images — otherwise its within-host
#    accuracy (and the Section-5 gate) is undefined for that disease.
empty = [d for d in diseases if (within_host_test["disease"] == d).sum() == 0]
assert not empty, (
    f"No within-host test images for {empty}. With the {PV_VARIANT!r} variant "
    "this can happen on a thin disease — try DEV_MODE = False, or revisit the "
    "disease list in NB00."
)

# A per-disease breakdown: training hosts kept, the host withheld, and the
# train/test counts on the training hosts.
print(f"{'disease':18s} {'train hosts':28s} {'withheld':10s} {'train':>7s} {'wh-test':>8s}")
for d in diseases:
    rows = train_hosts_frame[train_hosts_frame["disease"] == d]
    train_hosts = sorted(set(rows["host"]))
    n_train = (rows["split"] == "train").sum()
    n_test  = (rows["split"] == "test").sum()
    hosts_str = ", ".join(train_hosts)
    if len(hosts_str) > 27:
        hosts_str = hosts_str[:24] + "..."
    print(f"{d:18s} {hosts_str:28s} {held_out[d]:10s} {n_train:>7,} {n_test:>8,}")

### 3) Train the classifier (architecture per Rationale 2 setup)

Now train the classifier on the training pool you just built. This is a single call to `iri.train_baseline`, which handles the whole training procedure for you: it takes a ResNet-18 (a standard image classifier already trained on a large general image collection), replaces its final layer with a fresh one sized to your diseases, carves a small validation slice out of your training data to track progress, runs the training epochs, and hands back the version of the model that scored best on that validation slice.

Two things to keep in mind about what this call does and doesn't vary:

- **It trains with the library's standard image transform**, which includes mild, label-preserving augmentation — random crops and horizontal flips of the training images. That's the conventional way to train an image classifier, and it is *not* the thing under test in R2-Q4. Some other projects in this program deliberately vary the augmentation; this one varies the *host*. So the transform stays at its default and your attention stays on the hold-out.
- **The epoch count comes from `EPOCH_CAP`**, which DEV_MODE set. On a `DEV_MODE = True` run this is two epochs on a tiny slice — enough to confirm the notebook works, not enough to mean anything. The numbers you report come from the full run.

This is the step that needs a GPU and the step that takes real time — tens of minutes on the full run. The per-epoch log prints as it goes.

In [ ]:
# One call trains the classifier. train_baseline filters the frame to its
# `train` split internally (so the within-host test split you carved is never
# seen during training), carves its own validation slice for checkpointing,
# and returns the best-validation weights plus a per-epoch history.
#
# seed=42 makes the run reproducible: same data, same transform, same seed ->
# same model. EPOCH_CAP is the DEV_MODE lever (2 in dev, the full recipe in
# production). The train transform is left at its default — augmentation is not
# the variable in R2-Q4.
state_dict, history = iri.train_baseline(
    train_hosts_frame, hf, iri.PlantVillageDataset,
    num_classes=num_classes,
    epoch_cap=EPOCH_CAP,
    seed=42,
)

print(f"\nTraining done. Best validation accuracy: {history['best_val_acc']:.3f} "
      f"(epoch {history['best_val_epoch']}).")

Before moving on, look at the training curves below — they tell you whether training behaved.

The thing to watch for is the validation loss tracking the training loss downward rather than flattening or turning back up early. The validation accuracy climbing toward a plateau is the same signal from the other direction.

One point of vocabulary that's easy to trip on: the **validation accuracy** here is *not* the within-host accuracy you're going to report. The validation slice is a small piece carved out of the training images purely so training can decide when the model is at its best. The within-host accuracy — the number that matters, measured on PlantVillage's held-out test split — comes in the next section. Don't read `best_val_acc` as your result; it's a training-time signal, nothing more.

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))

ax_loss.plot(epochs, history["train_loss"], marker="o", label="train loss")
ax_loss.plot(epochs, history["val_loss"], marker="o", label="val loss")
ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("loss")
ax_loss.set_title("Training and validation loss")
ax_loss.legend()

ax_acc.plot(epochs, history["val_acc"], marker="o", color="tab:green")
ax_acc.axvline(history["best_val_epoch"], ls="--", color="gray",
               label=f"best epoch ({history['best_val_epoch']})")
ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("validation accuracy")
ax_acc.set_title("Validation accuracy")
ax_acc.legend()

plt.tight_layout()
plt.show()

if DEV_MODE:
    print("\nDEV_MODE is on — these curves only confirm the notebook runs. "
          "Two epochs on a tiny slice are not a trained model.")

### 4) Evaluate within-host: per-disease accuracy on the held-out within-host samples

You have a trained classifier. Now measure how well it does on the within-host test split — the held-out images from the same hosts it trained on.

The number that matters is **per-disease accuracy**: for each disease, of the test images that really are that disease, what fraction did the model label correctly? (This is sometimes called recall.) Per-disease is the right grain because the whole question is settled disease by disease — next week's cross-host number gets compared against this within-host number for the *same* disease, and "rust didn't transfer but bacterial spot did" is a per-disease claim. A single overall accuracy would average those apart and hide exactly the structure you're after.

The evaluation runs the model under a deterministic transform — no random crops or flips this time. Augmentation belongs to training, where it helps the model generalize; at evaluation you want the same image to give the same answer every run, so the number is stable.

In [ ]:
# Run the trained classifier over an evaluation slice and collect, for every
# image, its true disease and the model's predicted disease. This routine is
# written to be reused next week — the cross-host evaluation calls it with the
# withheld-host slice instead of the within-host test split; nothing else
# changes. (Both predictions and truth are disease indices, 0..N-1, because the
# model was trained on the disease label space — no remapping needed.)
import torch
from torch.utils.data import DataLoader

def predict(state_dict, eval_metadata, hf_dataset, *, num_classes, batch_size=32):
    model = iri.build_baseline_model(num_classes, pretrained=False)
    model.load_state_dict(state_dict)
    model = model.cuda().eval()

    ds = iri.PlantVillageDataset(
        eval_metadata, hf_dataset, transform=iri.imagenet_eval_transform()
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)

    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.cuda()).argmax(dim=1).cpu()
            y_pred.append(preds)
            y_true.append(labels)
    return torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()

idx_to_disease = {i: d for d, i in disease_to_idx.items()}
y_true, y_pred = predict(state_dict, within_host_test, hf, num_classes=num_classes)
print(f"Scored {len(y_true):,} within-host test images.")

**Practice 4.1 — compute per-disease accuracy.**

You have two arrays from the cell above: `y_true` (each image's real disease, as an index) and `y_pred` (the model's guess, same index space). Turn them into per-disease accuracy.

For each disease `d` with index `i`:

- Pick out the rows whose *true* label is `i` — those are the images that really are disease `d`.
- Among those rows, the per-disease accuracy is the fraction whose *predicted* label is also `i`.

Your fill is that one line. The loop, the row-selection, and the per-disease counts are given.

*Hint:* `is_this_disease` already marks the rows that really are disease `d`. The model's guesses for exactly those rows are `y_pred[is_this_disease]`. How many of them equal `i`?

In [ ]:
### 4.1) Per-disease accuracy
import numpy as np

# Overall accuracy: the fraction of all within-host test images classified
# correctly. A useful summary, but NOT the headline — the per-disease numbers
# below are.
overall_acc = float((y_pred == y_true).mean())

per_disease_acc = {}
per_disease_n = {}
for i, d in idx_to_disease.items():
    is_this_disease = (y_true == i)          # rows whose TRUE label is disease d
    per_disease_n[d] = int(is_this_disease.sum())

    # Practice 4.1 — replace this with the per-disease accuracy for disease d.
    raise NotImplementedError(
        "Practice 4.1 — set per_disease_acc[d] to the fraction of disease d's "
        "test rows the model predicted as d. See the hint in the markdown above."
    )
    # per_disease_acc[d] = ...

# The within-host accuracy report. Section 5 reads it; Section 6 saves it.
within_host_accuracy = {
    "overall": overall_acc,
    "per_disease": per_disease_acc,
    "n_per_disease": per_disease_n,
    "n_total": int(len(y_true)),
}
print("Per-disease accuracy computed.")

In [ ]:
# Read the report back as a table before moving on. Eyeball two things: every
# disease has a sensible number of test images behind its accuracy (a number
# computed on a handful of images is noisy), and the accuracies aren't all
# pinned at 0.0 or 1.0 (which usually means something upstream went wrong, not
# that the model is perfect).
print(f"{'disease':18s} {'within-host acc':>16s} {'n test':>8s}")
for d in diseases:
    print(f"{d:18s} {within_host_accuracy['per_disease'][d]:>16.3f} "
          f"{within_host_accuracy['n_per_disease'][d]:>8,}")
print(f"{'OVERALL':18s} {within_host_accuracy['overall']:>16.3f} "
      f"{within_host_accuracy['n_total']:>8,}")

### 5) Accuracy gate: is the within-host classifier good enough to be worth measuring transfer on?

You now have a within-host accuracy for each disease. Before next week's cross-host test, pause on one question: **is each within-host number high enough to be a meaningful reference?**

Here's why that matters, and it's the subtle point of this whole question. Next week you measure cross-host accuracy and compare it to this week's within-host accuracy, disease by disease. The gap is your finding. But that comparison only means something if the model actually learned the disease within host in the first place. If a disease sits near chance level even within host — if the model can barely tell it apart on the *same* hosts it trained on — then a small cross-host gap for that disease is not evidence of good transfer. There was never much skill there to transfer. The within-host number is the denominator of the whole finding, and a near-chance denominator makes the gap uninterpretable.

So this gate flags any disease whose within-host accuracy isn't clearly above chance. **Chance level** for an *N*-disease classifier is `1 / N` — what you'd get by guessing. The flag here is set at **twice chance** (`2 / N`): a disease at or below that is classified barely better than guessing, even within host, and gets flagged.

This is a flag, not a stop. Unlike a hard gate, nothing here halts the notebook or refuses to save your work. A flagged disease is still measured next week — the flag just tells you to read its cross-host gap with the right caveat instead of mistaking a small gap for successful transfer. The point is honest interpretation, not a pass/fail bar.

In [ ]:
# Chance level is 1/N; the flag is set at twice chance. A disease at or below
# the floor is classified barely better than guessing even within host, so its
# cross-host gap next week won't be interpretable as transfer. (This threshold
# is a judgment call, not a law — look at the actual numbers, not just the flag.)
WITHIN_HOST_FLOOR = 2 / num_classes

per_disease_flag = {}
for d in diseases:
    acc = within_host_accuracy["per_disease"][d]
    per_disease_flag[d] = "near-chance" if acc <= WITHIN_HOST_FLOOR else "ok"

flagged = [d for d in diseases if per_disease_flag[d] == "near-chance"]

# Fold the floor and the flags into the report so they're saved and carried
# forward to next week.
within_host_accuracy["null_floor"] = float(1 / num_classes)
within_host_accuracy["flag_threshold"] = float(WITHIN_HOST_FLOOR)
within_host_accuracy["per_disease_flag"] = per_disease_flag

print(f"Chance level: {1/num_classes:.3f}   Flag threshold (2x chance): "
      f"{WITHIN_HOST_FLOOR:.3f}\n")
print(f"{'disease':18s} {'within-host acc':>16s} {'flag':>14s}")
for d in diseases:
    print(f"{d:18s} {within_host_accuracy['per_disease'][d]:>16.3f} "
          f"{per_disease_flag[d]:>14s}")

if flagged:
    print(f"\n{len(flagged)} disease(s) flagged near-chance: {', '.join(flagged)}")
    print("These are still measured next week — read their gaps with caution.")
else:
    print("\nNo diseases flagged. Every within-host number is a sound reference "
          "for next week's comparison.")

if DEV_MODE:
    print("\nDEV_MODE is on — these flags reflect a two-epoch dev run, not a "
          "real assessment. Re-run with DEV_MODE = False before reading them.")

**Write the interpretation.**

The flags above are computed for you. What isn't computed — and what only you can write — is what they *mean* for your Week-3 reading. Fill in the `interpretation` field in the cell below.

If any diseases were flagged, cover, for each one:

- that its within-host accuracy is near chance, so the model barely classifies it even on familiar hosts;
- what that does to next week's gap — a small cross-host gap there would *not* be evidence that the disease transferred, because there was little within-host skill to transfer in the first place;
- whether you'd still report that disease, and with what caveat.

If nothing was flagged, say so, and note that the within-host numbers are a sound reference for the cross-host comparison.

This isn't busywork — it's the sentence that keeps you from over-claiming in Week 4. A flagged disease that quietly shows a small gap could otherwise read as a transfer success when it's really a classification failure.

In [ ]:
### 5.1) Your interpretation of the within-host gate
#
# Edit the `interpretation` string below. The structural fields are filled in
# for you from the flags above; the interpretation is yours to write.
gate = {
    "within_host_floor": float(WITHIN_HOST_FLOOR),
    "flagged_diseases": flagged,
    "interpretation": (
        # Cover at minimum (see the markdown above):
        #  - for each flagged disease: that its within-host accuracy is near
        #    chance, and what that means for reading its cross-host gap next week;
        #  - whether you'd still report each flagged disease, and with what caveat;
        #  - if nothing was flagged: say so, and that the within-host numbers are
        #    a sound reference for the comparison.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Soft check — a reminder, not a stop. The notebook saves your work either way;
# this just flags an interpretation you haven't written yet.
if gate["interpretation"].strip().startswith("PLACEHOLDER"):
    print("REMINDER: the within-host gate interpretation is still the "
          "placeholder. Rewrite gate['interpretation'] before you submit EQ#2.")
else:
    print("Within-host gate interpretation recorded.")

### 6) Closeout: save `classifier.pkl`, `within_host_test.parquet`, and `within_host_accuracy.parquet`; submit EQ#2

Week 2 is done once this notebook has saved four things. The first is the classifier itself; the other three are the record that makes this week reproducible and feeds next week's cross-host test.

- `classifier.pt` — the trained weights (the state_dict `train_baseline` returned), saved with `torch.save`. NB02 reloads it next week to run on the withheld host.
- `within_host_accuracy.parquet` — per-disease within-host accuracy plus an overall row, carrying the chance floor and the gate flag for each disease.
- `within_host_test.parquet` — exactly which images formed the within-host test split, so the numbers above can be reproduced.
- `within_host_summary.json` — a compact record of the results, how they were produced, and your gate interpretation. NB03 reads your interpretation back when it builds the Week-4 finding.

In [ ]:
import torch

# Save the trained weights. train_baseline returned a state_dict (not a model),
# and torch.save writes it straight to disk. NB02 reloads it the way `predict`
# did: build_baseline_model(num_classes, pretrained=False) then load_state_dict.
# num_classes is reproducible next week from the precommitted disease list
# (enumerate it the same way Section 2 did). The .pt extension is the PyTorch
# convention for a state dict.
classifier_path = OUTPUT_DIR / "classifier.pt"
torch.save(state_dict, classifier_path)
print(f"Wrote {classifier_path}")

In [ ]:
# Build the within-host accuracy table: one row per disease plus an overall row.
# The chance floor and the gate flag travel with each disease so next week's
# comparison can honor them without recomputing. Reload it once to confirm it
# round-tripped.
import pandas as pd

rows = []
for d in diseases:
    rows.append({
        "disease":         d,
        "within_host_acc": float(within_host_accuracy["per_disease"][d]),
        "n_test":          int(within_host_accuracy["n_per_disease"][d]),
        "null_floor":      float(within_host_accuracy["null_floor"]),
        "flag_threshold":  float(within_host_accuracy["flag_threshold"]),
        "flag":            within_host_accuracy["per_disease_flag"][d],
    })
rows.append({
    "disease":         "overall",
    "within_host_acc": float(within_host_accuracy["overall"]),
    "n_test":          int(within_host_accuracy["n_total"]),
    "null_floor":      float(within_host_accuracy["null_floor"]),
    "flag_threshold":  float(within_host_accuracy["flag_threshold"]),
    "flag":            "",
})
report_df = pd.DataFrame(
    rows,
    columns=["disease", "within_host_acc", "n_test",
             "null_floor", "flag_threshold", "flag"],
)
report_path = OUTPUT_DIR / "within_host_accuracy.parquet"
report_df.to_parquet(report_path, index=False)

reloaded = pd.read_parquet(report_path)
assert "overall" in set(reloaded["disease"]), "report has no overall row"
assert set(reloaded.columns) == set(report_df.columns), "report column drift"
print(f"Wrote {report_path}  ({len(report_df)} rows)")

In [ ]:
# The within-host test split, so the report is reproducible: which exact images
# were scored. Keep the index — it's the alignment key into the images. Then a
# compact summary in the R2 results / methodology / interpretation shape, which
# also records the label order and num_classes NB02 needs to rebuild the model,
# and carries your gate interpretation forward to NB03. Floats are cast so the
# JSON serializes.
import json

within_host_test.to_parquet(OUTPUT_DIR / "within_host_test.parquet")

summary = {
    "results": {
        "overall_within_host_acc": float(within_host_accuracy["overall"]),
        "per_disease_within_host_acc": {
            d: float(within_host_accuracy["per_disease"][d]) for d in diseases
        },
        "per_disease_flag": within_host_accuracy["per_disease_flag"],
        "flagged_diseases": gate["flagged_diseases"],
        "n_per_disease": {d: int(within_host_accuracy["n_per_disease"][d])
                          for d in diseases},
    },
    "methodology": {
        "label_space": "disease",
        "diseases": diseases,            # the label order, for NB02 to rebuild
        "num_classes": num_classes,
        "hold_out_scheme": scheme,
        "held_out_host": held_out,
        "null_floor": float(within_host_accuracy["null_floor"]),
        "flag_threshold": float(within_host_accuracy["flag_threshold"]),
        "dev_mode": DEV_MODE,            # implemented, not committed
        "epoch_cap": EPOCH_CAP,
        "best_val_acc": float(history["best_val_acc"]),
    },
    "interpretation": gate["interpretation"],
}

summary_path = OUTPUT_DIR / "within_host_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

with open(summary_path) as f:
    reloaded = json.load(f)
assert set(reloaded.keys()) == {"results", "methodology", "interpretation"}
assert reloaded["methodology"]["diseases"] == diseases, "label order drift"
print(f"Wrote {OUTPUT_DIR / 'within_host_test.parquet'}")
print(f"Wrote {summary_path}")

**Submit EQ#2.**

Your Week-2 deliverable is the EQ#2 report — your own writeup, not a file this notebook produces. The notebook gives you the evidence base; EQ#2 is where you explain it. Build your answer from what you produced here:

- the within-host accuracy for each disease, and the overall figure;
- which diseases (if any) the gate flagged as near-chance, and the interpretation you wrote for them — that's the caveat that keeps next week's gaps honest;
- what your within-host results lead you to expect when you measure cross-host accuracy in Week 3, disease by disease.

The four files above stay in your output directory. NB02 reloads `classifier.pt` to run on the withheld host, and reads your hold-out scheme back from `precommit.json`, so next week's cross-host test is carved from the same decisions as this week's within-host test. The gap between the two — per disease — is the finding your Week-4 paper is built on.